# 04b - Rerun Missing inferCNV Datasets

This notebook reruns only datasets listed in:
`results/plots/summary/runs_missing_metrics.csv`

It is designed to be restart-safe:
- updates `manifests/runs.csv` after each run
- appends to `results/logs/run_status.csv`
- skips existing `_CNVinf.h5ad` unless overwrite is enabled


In [1]:
from pathlib import Path
import time
import pandas as pd
import scanpy as sc
import sys

sys.path.append('/home/augusta/storage3/augusta/insituCNV/InSituCNV')
import insitucnv as icv


In [2]:
ROOT = Path('/home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains')

RUNS_MANIFEST = ROOT / 'manifests' / 'runs.csv'
RUNS_DIR = ROOT / 'data' / 'runs'
STATUS_LOG = ROOT / 'results' / 'logs' / 'run_status.csv'
MISSING_CSV = ROOT / 'results' / 'plots' / 'summary' / 'runs_missing_metrics.csv'

PLOT_DIR = ROOT / 'results' / 'plots' / 'infercnv'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

assert RUNS_MANIFEST.exists(), f'Missing runs manifest: {RUNS_MANIFEST}'
assert MISSING_CSV.exists(), f'Missing missing-runs csv: {MISSING_CSV}'
assert RUNS_DIR.exists(), f'Missing runs directory: {RUNS_DIR}'

print(f'RUNS_MANIFEST: {RUNS_MANIFEST}')
print(f'MISSING_CSV: {MISSING_CSV}')
print(f'RUNS_DIR: {RUNS_DIR}')
print(f'PLOT_DIR: {PLOT_DIR}')


RUNS_MANIFEST: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/manifests/runs.csv
MISSING_CSV: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/results/plots/summary/runs_missing_metrics.csv
RUNS_DIR: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/data/runs
PLOT_DIR: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/results/plots/infercnv


## Controls

- `OVERWRITE_CNVINF=False`: skip runs where `_CNVinf.h5ad` already exists.
- `ONLY_FAILED_OR_MISSING=True`: rerun only rows that were `failed` or `missing_input`.


In [3]:
OVERWRITE_CNVINF = False
ONLY_FAILED_OR_MISSING = True
MAX_RESOLUTION_STEPS = 100


def log_status(run_id, template_id, count_fraction, gene_panel, stage, status, duration_sec=0.0, message=''):
    now = pd.Timestamp.now()
    row = pd.DataFrame([{
        'run_id': run_id,
        'template_id': template_id,
        'count_fraction': count_fraction,
        'gene_panel': gene_panel,
        'stage': stage,
        'status': status,
        'start_time': now,
        'end_time': now,
        'duration_sec': duration_sec,
        'message': message,
    }])
    if STATUS_LOG.exists():
        row.to_csv(STATUS_LOG, mode='a', header=False, index=False)
    else:
        row.to_csv(STATUS_LOG, index=False)


## Build Rerun Candidate List


In [4]:
runs_df = pd.read_csv(RUNS_MANIFEST)
missing_df = pd.read_csv(MISSING_CSV)

runs_df['run_id'] = runs_df['run_id'].astype(str)
missing_df['run_id'] = missing_df['run_id'].astype(str)

if ONLY_FAILED_OR_MISSING and 'status' in missing_df.columns:
    m = missing_df['status'].astype(str).str.lower()
    missing_df = missing_df[m.isin(['failed', 'missing_input'])].copy()

cand = runs_df[runs_df['run_id'].isin(set(missing_df['run_id']))].copy()

if 'status' in cand.columns:
    print('Candidate status counts before rerun:')
    print(cand['status'].astype(str).value_counts(dropna=False).to_string())

print(f'Total rerun candidates: {len(cand)}')
display(cand[['run_id', 'template_id', 'count_fraction', 'gene_panel', 'status']].head(20))


Candidate status counts before rerun:
status
failed    6
Total rerun candidates: 6


,run_id,template_id,count_fraction,gene_panel,status
115,T02_C70_G20000,T02,70,20000,failed
222,T04_C10_G5000,T04,10,5000,failed
343,T06_C10_G1000,T06,10,1000,failed
352,T06_C20_G15000,T06,20,15000,failed
370,T06_C70_Gall,T06,70,all,failed
613,T10_C50_G500,T10,50,500,failed


## Rerun inferCNV For Missing Runs


In [15]:
FORCE_RUN_IDS = {'T02_C70_G20000', 'T04_C10_G5000', 'T06_C10_G1000', 'T06_C20_G15000', 'T06_C70_Gall', 'T10_C50_G500'
}

cand = cand[cand['run_id'].astype(str).isin(FORCE_RUN_IDS)].copy()
OVERWRITE_CNVINF = True


In [16]:
updated = runs_df.copy()

for r in cand.itertuples(index=False):
    run_id = str(r.run_id)
    in_path = RUNS_DIR / f'{run_id}.h5ad'
    out_path = RUNS_DIR / f'{run_id}_CNVinf.h5ad'

    if not in_path.exists():
        msg = f'missing input {in_path.name}'
        print(f'SKIP {run_id}: {msg}')
        updated.loc[updated['run_id'].astype(str) == run_id, 'status'] = 'missing_input'
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'infercnv_rerun', 'missing_input', 0.0, msg)
        updated.to_csv(RUNS_MANIFEST, index=False)
        continue

    if out_path.exists() and not OVERWRITE_CNVINF:
        print(f'SKIP {run_id}: output exists')
        updated.loc[updated['run_id'].astype(str) == run_id, 'status'] = 'inferred'
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'infercnv_rerun', 'success', 0.0, 'already exists')
        updated.to_csv(RUNS_MANIFEST, index=False)
        continue

    t0 = time.time()
    try:
        icv.tl.inferCNV_for_simulated_data(
            path=str(RUNS_DIR),
            data_name=run_id,
            window_size=int(r.window_size),
            plot_dir=str(PLOT_DIR),
            max_resolution_steps=int(MAX_RESOLUTION_STEPS),
        )

        dt = round(time.time() - t0, 2)
        updated.loc[updated['run_id'].astype(str) == run_id, 'status'] = 'inferred'
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'infercnv_rerun', 'success', dt, f'output={out_path.name}')
        print(f'OK {run_id} ({dt}s)')
    except Exception as e:
        dt = round(time.time() - t0, 2)
        updated.loc[updated['run_id'].astype(str) == run_id, 'status'] = 'failed'
        log_status(run_id, r.template_id, int(r.count_fraction), str(r.gene_panel), 'infercnv_rerun', 'failed', dt, str(e))
        print(f'FAILED {run_id}: {e}')

    # Persist after each run to survive kernel/interruption
    updated.to_csv(RUNS_MANIFEST, index=False)

print(f'Updated runs manifest: {RUNS_MANIFEST}')


Updated runs manifest: /home/augusta/storage3/augusta/insituCNV/InSituCNV/Evaluate_technical_constrains/manifests/runs.csv


## Post-Rerun Check

After this, rerun `05_compute_metrics_batch.ipynb` to fill metrics for newly inferred runs.


In [17]:
r = pd.read_csv(RUNS_MANIFEST)
if 'status' in r.columns:
    print(r['status'].astype(str).value_counts(dropna=False).to_string())

missing_now = set(pd.read_csv(MISSING_CSV)['run_id'].astype(str)) - set(
    r.loc[r['status'].astype(str).str.lower() == 'inferred', 'run_id'].astype(str)
)
print(f'Still not inferred among originally missing: {len(missing_now)}')
print('Examples:', sorted(list(missing_now))[:15])


status
inferred    621
failed        9
Still not inferred among originally missing: 6
Examples: ['T02_C70_G20000', 'T04_C10_G5000', 'T06_C10_G1000', 'T06_C20_G15000', 'T06_C70_Gall', 'T10_C50_G500']
